# Week 7 skills notebook: joining two tables

This notebook belongs to the skills track. It uses a made-up bakery with four shops, so that the skills stay separate from this week's project.

The bakery keeps its records in separate tables: one for its shops, one for its monthly sales and one for its bread prices. Most questions about the bakery need two of them at once. You will join tables, predict and count the rows of every join, catch a join that loses rows without a warning, and reshape a table that arrives in the wrong layout.

**How to use it:** run each cell once, in order. Before every join, write down how many rows you expect, then compare your guess with the count.

## 1. Three tables

The cell below makes the bakery's three tables. The seed in `default_rng(77)` makes the random sales the same for everyone.

- `shops` has one row per shop, with the district the shop is in.
- `sales` has one row per shop per month, with the number of loaves the shop sold.
- `prices` has one row per month, with the price of a loaf. Nobody has entered December's price yet.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(77)

shops = pd.DataFrame({"shop": ["S01", "S02", "S03", "S04"], "district": [1, 1, 2, 3]})

months = list(range(1, 13))
sales = pd.DataFrame([(s, m) for s in shops["shop"] for m in months], columns=["shop", "month"])
sales["loaves"] = rng.poisson(900, size=len(sales))

# one price per month; December has not been entered yet
prices = pd.DataFrame({"month": months[:11], "price": np.round(np.linspace(2.00, 2.50, 11), 2)})

print("shops:", shops.shape, " sales:", sales.shape, " prices:", prices.shape)
sales.head()

## 2. What one row means

Every table has a grain: the thing that one row describes. In `shops` one row is a shop. In `sales` one row is a shop in a month. In `prices` one row is a month.

The key is the column, or the set of columns, that names that thing. A join matches rows by their key, so test the key before you join. A key that should appear once must appear once.

In [ ]:
print("Each shop appears once in shops:", shops["shop"].is_unique)
print("Each month appears once in prices:", prices["month"].is_unique)
print("Repeated shop and month pairs in sales:", sales.duplicated(["shop", "month"]).sum())

## 3. A first join

To total the loaves sold in each district, you need the district from `shops` next to every row of `sales`. A left join keeps every row of the left table (`sales`) and adds the matching columns from the right table (`shops`).

Each shop appears once in `shops`, so every sales row finds exactly one partner. The joined table should therefore have as many rows as `sales`. Predict the number, then run the cell.

In [ ]:
with_district = sales.merge(shops, on="shop", how="left")
print("Rows before the join:", len(sales))
print("Rows after the join:", len(with_district))
assert len(with_district) == len(sales)
print()
print(with_district.groupby("district")["loaves"].sum())

## 4. Four kinds of join

Two small tables make the four kinds easy to see. The first has loaves sold for months 1 to 5. The second has prices for months 3 to 7. Only months 3, 4 and 5 are in both.

- An **inner** join keeps only the rows whose key is in both tables.
- A **left** join keeps every row of the left table, with blanks where the right table has no match.
- A **right** join keeps every row of the right table.
- An **outer** join keeps every row of both tables.

Predict the number of rows for each kind before you run the cell.

In [ ]:
small_sales = pd.DataFrame({"month": [1, 2, 3, 4, 5], "loaves": [880, 905, 870, 930, 910]})
small_prices = pd.DataFrame({"month": [3, 4, 5, 6, 7], "price": [2.10, 2.15, 2.20, 2.25, 2.30]})

for how in ["inner", "left", "right", "outer"]:
    joined = small_sales.merge(small_prices, on="month", how=how)
    print(f"{how} join: {len(joined)} rows, months {joined['month'].tolist()}")

An outer join with `indicator=True` adds a column called `_merge`. It says where each row came from: `both`, `left_only` or `right_only`. It is the quickest way to see what a join matched and what it could not match.

In [ ]:
check = small_sales.merge(small_prices, on="month", how="outer", indicator=True)
print(check)
print()
print(check["_merge"].value_counts())

## 5. Read code an AI assistant wrote

We asked an AI assistant: *"Join the sales to the prices and give me the bakery's total revenue for the year."* It wrote the cell below. Run it, then decide whether the answer is right.

In [ ]:
# Written by an AI assistant
priced = sales.merge(prices, on="month")
priced["revenue"] = priced["loaves"] * priced["price"]
print("Total revenue for the year:", f"{priced['revenue'].sum():,.2f}")

The code runs without an error, and the total looks reasonable. Now count the rows. `merge` does an inner join unless you ask for another kind, and an inner join keeps only the rows that match in both tables.

In [ ]:
print("Rows before the join:", len(sales))
print("Rows after the join:", len(priced))
print("Months in the result:", priced["month"].nunique())

Four rows disappeared without a warning. They are the December sales of all four shops, and they vanished because `prices` has no row for December. The AI's total covers eleven months, while its label says a year.

The next cell does the join the careful way. A left join keeps every sales row. The indicator shows which rows found no price. The `validate` argument makes pandas stop if a month appears twice in `prices`.

In [ ]:
careful = sales.merge(prices, on="month", how="left", indicator=True, validate="many_to_one")
no_price = careful[careful["_merge"] == "left_only"]
print("Rows with no price:", len(no_price))
print("Loaves sold in December with no price:", f"{no_price['loaves'].sum():,}")
careful["revenue"] = careful["loaves"] * careful["price"]
print("Revenue for January to November:", f"{careful['revenue'].sum():,.2f}")

# CHECK: every sales row survives the careful join, and only December lacks a price
assert len(careful) == len(sales)
assert set(no_price["month"]) == {12}
print("CHECK passed")

The revenue is the same number the AI printed. The difference is that you now know what it covers, and you can say so: revenue for January to November, with December's sales waiting for a price. A report that says this is honest. A report that calls the same number a year's revenue misleads its reader.

## 6. When a key repeats

Suppose someone corrects the June price by adding a new row instead of changing the old one. June now appears twice in the price table, and every June sales row finds two partners. The join keeps both, so June is counted twice.

In [ ]:
prices_twice = pd.concat([prices, pd.DataFrame({"month": [6], "price": [2.30]})], ignore_index=True)
doubled = sales.merge(prices_twice, on="month", how="left")
doubled["revenue"] = doubled["loaves"] * doubled["price"]
print("Rows before the join:", len(sales))
print("Rows after the join with June twice:", len(doubled))
print("Revenue with June counted twice:", f"{doubled['revenue'].sum():,.2f}")

The row count gives the error away again: more rows came out than went in. A left join on a key that appears once on the right can never add rows, so extra rows always mean a repeated key.

`validate="many_to_one"` states that promise in code: many sales rows per month, one price per month. When the price table breaks the promise, pandas stops instead of guessing.

In [ ]:
try:
    sales.merge(prices_twice, on="month", how="left", validate="many_to_one")
except pd.errors.MergeError as err:
    print("pandas stopped the join:", err)

The repair belongs in the price table. Someone has to decide which June price is right. Here we keep the later row, then join again.

In [ ]:
prices_fixed = prices_twice.drop_duplicates("month", keep="last")
fixed = sales.merge(prices_fixed, on="month", how="left", validate="many_to_one")
print("Rows after the join with the repaired prices:", len(fixed))

## 7. Keys that look the same but do not match

A join compares keys exactly. Two keys that look the same to a person can differ for the computer in three common ways: a space at the end, a different use of capitals, or a number stored as text.

A manager typed this table of bakers per shop by hand. Count what matches.

In [ ]:
bakers = pd.DataFrame({"shop": ["S01", "s02", "S03 ", "S04"], "bakers": [5, 3, 4, 2]})
matched = shops.merge(bakers, on="shop", how="inner")
print("Shops:", len(shops), " rows in bakers:", len(bakers), " matched:", len(matched))
print("codes in shops: ", shops["shop"].tolist())
print("codes in bakers:", bakers["shop"].tolist())

Two shops vanished from the inner join. The code `s02` is written in lower case, and `S03 ` ends with a space. Clean the codes in the same way in both tables before you join.

In [ ]:
bakers["shop"] = bakers["shop"].str.strip().str.upper()
matched = shops.merge(bakers, on="shop", how="inner")
print("Matched after cleaning the codes:", len(matched))

A number stored as text fails more loudly. Many web sources send every value as text, so a month can arrive as `"6"` rather than `6`. pandas refuses to join a text key to a number key. Read its message to the end.

In [ ]:
prices_text = prices.astype({"month": str})      # the month as text, the way many web sources send it
try:
    sales.merge(prices_text, on="month", how="left")
except ValueError as err:
    print("pandas refused:", err)

The message suggests `pd.concat`, which is the wrong tool for this job. `concat` stacks tables, or places them side by side by row position, and it never looks at the key. The cell below shows the damage on a small case: four months of sales from the first month on, placed next to four prices sorted from the latest month back.

In [ ]:
first_rows = sales[sales["shop"] == "S01"].head(4).reset_index(drop=True)
newest_first = prices.sort_values("month", ascending=False).head(4).reset_index(drop=True)
side_by_side = pd.concat([first_rows, newest_first], axis=1)
print(side_by_side)

Every row now pairs a month with another month's price. The repair is to convert the text to a number, then join on the key.

In [ ]:
prices_text["month"] = prices_text["month"].astype(int)
joined = sales.merge(prices_text, on="month", how="left", validate="many_to_one")
print("Rows after converting and joining:", len(joined))

## 8. Wide and long tables

The same numbers can be laid out in two ways, and the layout decides which jobs are easy.

- A **wide** table has one row per thing and one column per month or year. A person can read it easily, and many spreadsheets and downloads arrive this way.
- A **long** table has one row per thing per month or year. It is easy to join, filter and compute with.

The bakery's monthly sales targets arrive wide: one row per shop, one column per month.

In [ ]:
targets_wide = pd.DataFrame({"shop": shops["shop"]})
for m in months:
    targets_wide[str(m)] = rng.choice([850, 900, 950], size=len(shops))   # column names are text, as in a downloaded file
targets_wide

`melt` turns the wide table into a long one. The column names become values in a new column. Column names are always text, so the new month column is text too, and it must become a number before it can be joined to `sales`.

In [ ]:
targets = targets_wide.melt(id_vars="shop", var_name="month", value_name="target")
print(targets.dtypes)
targets["month"] = targets["month"].astype(int)
print()
print("Rows in the long table:", len(targets))
targets.head()

The long table has the same grain as `sales`: one row per shop per month. Its key is two columns, `shop` and `month`, and a join on both is written `on=["shop", "month"]`.

## 9. Your turn: did each shop meet its target?

Join `targets` to `sales` on both `shop` and `month`. Then make a Series called `hits`, with one value per shop: the number of months in which the shop sold at least its target. Include a shop even if its count is zero.

Ask your AI assistant for the code. Before you run it, write down how many rows the join should give, and check the count afterwards.

In [ ]:
# your turn 1: make a Series called hits, indexed by shop, with the number of months each shop met its target
hits = None

In [ ]:
# CHECK 1
assert hits is not None, "hits is still empty"
expected = {}
for s in shops["shop"]:
    n = 0
    for m in months:
        sold = sales.loc[(sales["shop"] == s) & (sales["month"] == m), "loaves"].iloc[0]
        goal = targets.loc[(targets["shop"] == s) & (targets["month"] == m), "target"].iloc[0]
        n += int(sold >= goal)
    expected[s] = n
assert {k: int(v) for k, v in hits.items()} == expected, "the counts do not match"
print("CHECK 1 passed:", expected)

## 10. Your turn: revenue by district

Now put the week together. Make two things:

- `district_revenue`, a Series indexed by district, with the revenue from January to November;
- `unpriced`, the number of sales rows that have no price.

Use joins that keep every sales row, add `validate` to both joins, and print the row count after each one. Ask your AI assistant for help, and check every join it writes by its row count.

In [ ]:
# your turn 2: make district_revenue (a Series indexed by district) and unpriced (a whole number)
district_revenue = None
unpriced = None

In [ ]:
# CHECK 2
assert district_revenue is not None and unpriced is not None, "district_revenue or unpriced is still empty"
price_of = dict(zip(prices["month"], prices["price"]))
by_district = {}
for r in sales.merge(shops, on="shop").itertuples():
    if r.month in price_of:
        by_district[r.district] = by_district.get(r.district, 0) + r.loaves * price_of[r.month]
assert set(int(k) for k in district_revenue.index) == set(by_district), "one row per district is needed"
assert all(abs(float(district_revenue[k]) - v) < 0.01 for k, v in by_district.items()), "the revenue does not add up"
assert int(unpriced) == int((~sales["month"].isin(prices["month"])).sum()), "unpriced is not the number of rows without a price"
print("CHECK 2 passed: revenue adds up to", f"{float(district_revenue.sum()):,.2f}", "and", int(unpriced), "rows have no price")

## 11. The habit for every join

1. Say what one row means in each table, and test that the key is unique where it should be.
2. Check that the keys have the same type and the same spelling in both tables.
3. Choose the kind of join from the question, and predict the number of rows.
4. Run the join with `validate`, and with `indicator=True` when some rows might not match.
5. Count the rows after the join. If the count differs from your prediction, find out why before you compute anything.